# 02 - Baseline ML pentru RUL: strict benchmark si clean benchmark

Acest notebook construieste un baseline reproductibil pentru predictia RUL si ruleaza doua scenarii:

- **`all_eligible`**: toate bateriile cu cel putin 20 cicluri utile. Este testul strict de generalizare.
- **`clean_benchmark`**: baterii cu istoric suficient si curbe de capacitate/SOH mai coerente. Este scenariul principal recomandat pentru grafice si comparatie in lucrare.

Modele:

- `SVR_RBF`
- `RandomForest`
- `HistGradientBoosting`

Toate spliturile sunt pe `battery_id`, nu pe randuri.

## 1. Imports si paths

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="Could not find the number of physical cores")

SEED = 42
CLEAN_SPLIT_SEED = 7
np.random.seed(SEED)

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

DATASET_ROOT = REPO_ROOT / "data_test" / "Cleaned Datasets" / "Datasets 5-56 cleaned" / "cleaned_dataset"
METADATA_PATH = DATASET_ROOT / "metadata.csv"
CYCLE_DATA_DIR = DATASET_ROOT / "data"

ARTIFACTS_DIR = REPO_ROOT / "artifacts"
FEATURE_DIR = ARTIFACTS_DIR / "features"
SPLIT_DIR = ARTIFACTS_DIR / "splits"
MODEL_DIR = ARTIFACTS_DIR / "models"
METRICS_DIR = ARTIFACTS_DIR / "metrics"
PRED_DIR = ARTIFACTS_DIR / "predictions"
FIG_DIR = ARTIFACTS_DIR / "figures" / "baselines"
TABLE_DIR = ARTIFACTS_DIR / "tables"
for d in [FEATURE_DIR, SPLIT_DIR, MODEL_DIR, METRICS_DIR, PRED_DIR, FIG_DIR, TABLE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

FEATURE_TABLE_PATH = FEATURE_DIR / "battery_cycle_features_v2.csv"
FEATURE_COLUMNS_PATH = FEATURE_DIR / "baseline_feature_columns_v2.json"
SCENARIO_CONFIG_PATH = SPLIT_DIR / "modeling_scenarios_v1.json"
QUALITY_TABLE_PATH = TABLE_DIR / "battery_modeling_quality.csv"

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["figure.figsize"] = (12, 6)

assert METADATA_PATH.exists(), f"Missing metadata file: {METADATA_PATH}"
assert CYCLE_DATA_DIR.exists(), f"Missing cycle data directory: {CYCLE_DATA_DIR}"

print("Repository:", REPO_ROOT)
print("Feature table:", FEATURE_TABLE_PATH)
print("Scenario config:", SCENARIO_CONFIG_PATH)

## 2. Functii pentru parsare si feature extraction

In [ ]:
def parse_start_time(value) -> pd.Timestamp:
    if pd.isna(value):
        return pd.NaT
    numbers = np.fromstring(str(value).replace("[", " ").replace("]", " ").replace(",", " "), sep=" ")
    if numbers.size < 6:
        return pd.NaT
    year, month, day, hour, minute, second = numbers[:6]
    sec_int = int(second)
    micro = int(round((float(second) - sec_int) * 1_000_000))
    try:
        return pd.Timestamp(year=int(year), month=int(month), day=int(day), hour=int(hour), minute=int(minute), second=sec_int, microsecond=micro)
    except ValueError:
        return pd.NaT


def parse_numeric(value) -> float:
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float, np.number)):
        return float(value)
    import re
    match = re.search(r"[-+]?(?:\d*\.\d+|\d+)(?:[eE][-+]?\d+)?", str(value))
    if not match:
        return np.nan
    try:
        return float(match.group(0))
    except ValueError:
        return np.nan


def safe_slope(x_values: np.ndarray, y_values: np.ndarray) -> float:
    x = np.asarray(x_values, dtype=float)
    y = np.asarray(y_values, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    if x.size < 2:
        return np.nan
    x_centered = x - x.mean()
    denom = float(np.dot(x_centered, x_centered))
    if denom == 0:
        return 0.0
    return float(np.dot(x_centered, y - y.mean()) / denom)


def summarize_signal(series: pd.Series, prefix: str, name: str) -> dict[str, float]:
    values = pd.to_numeric(series, errors="coerce").to_numpy(dtype=float)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return {f"{prefix}{name}_mean": np.nan, f"{prefix}{name}_std": np.nan, f"{prefix}{name}_min": np.nan, f"{prefix}{name}_max": np.nan}
    return {
        f"{prefix}{name}_mean": float(np.mean(values)),
        f"{prefix}{name}_std": float(np.std(values)),
        f"{prefix}{name}_min": float(np.min(values)),
        f"{prefix}{name}_max": float(np.max(values)),
    }


def extract_cycle_features(csv_path: Path, prefix: str) -> dict[str, float]:
    cycle = pd.read_csv(csv_path)
    result: dict[str, float] = {"filename": csv_path.name}

    column_map = {
        "Voltage_measured": "voltage",
        "Current_measured": "current",
        "Temperature_measured": "temperature",
        "Current_load": "current_load",
        "Voltage_load": "voltage_load",
        "Current_charge": "current_charge",
        "Voltage_charge": "voltage_charge",
    }
    for raw_col, short_name in column_map.items():
        if raw_col in cycle.columns:
            result.update(summarize_signal(cycle[raw_col], prefix, short_name))

    if "Time" in cycle.columns:
        time = pd.to_numeric(cycle["Time"], errors="coerce").to_numpy(dtype=float)
        time = time[np.isfinite(time)]
        result[f"{prefix}time_duration"] = float(np.max(time) - np.min(time)) if time.size else np.nan
    else:
        result[f"{prefix}time_duration"] = np.nan

    if {"Time", "Voltage_measured", "Current_measured"}.issubset(cycle.columns):
        time = pd.to_numeric(cycle["Time"], errors="coerce").to_numpy(dtype=float)
        voltage = pd.to_numeric(cycle["Voltage_measured"], errors="coerce").to_numpy(dtype=float)
        current = pd.to_numeric(cycle["Current_measured"], errors="coerce").to_numpy(dtype=float)
        mask = np.isfinite(time) & np.isfinite(voltage) & np.isfinite(current)
        if mask.sum() >= 2:
            t = time[mask]
            v = voltage[mask]
            i = current[mask]
            order = np.argsort(t)
            t = t[order]
            v = v[order]
            i = i[order]
            result[f"{prefix}energy_wh"] = float(np.trapezoid(np.abs(v * i), t) / 3600.0)
            result[f"{prefix}charge_ah"] = float(np.trapezoid(np.abs(i), t) / 3600.0)
            result[f"{prefix}voltage_slope"] = safe_slope(t, v)
            if "Temperature_measured" in cycle.columns:
                temp = pd.to_numeric(cycle["Temperature_measured"], errors="coerce").to_numpy(dtype=float)[mask][order]
                result[f"{prefix}temperature_slope"] = safe_slope(t, temp)
            else:
                result[f"{prefix}temperature_slope"] = np.nan
        else:
            result[f"{prefix}energy_wh"] = np.nan
            result[f"{prefix}charge_ah"] = np.nan
            result[f"{prefix}voltage_slope"] = np.nan
            result[f"{prefix}temperature_slope"] = np.nan

    return result


def rolling_slope(values: pd.Series, window: int = 5) -> pd.Series:
    return values.rolling(window=window, min_periods=3).apply(lambda x: safe_slope(np.arange(len(x)), x), raw=False)


def regression_metrics(y_true, y_pred) -> dict[str, float]:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return {
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "R2": float(r2_score(y_true, y_pred)),
    }


def savefig(name: str):
    path = FIG_DIR / name
    plt.savefig(path, dpi=180, bbox_inches="tight")
    print(f"Saved: {path}")

## 3. Construire sau incarcare feature table

In [ ]:
def build_feature_table() -> pd.DataFrame:
    metadata = pd.read_csv(METADATA_PATH)
    metadata["start_time"] = metadata["start_time"].apply(parse_start_time)
    metadata["Capacity"] = metadata["Capacity"].apply(parse_numeric)
    metadata["Re"] = metadata["Re"].apply(parse_numeric)
    metadata["Rct"] = metadata["Rct"].apply(parse_numeric)
    metadata["ambient_temperature"] = metadata["ambient_temperature"].apply(parse_numeric)
    metadata["filename"] = metadata["filename"].astype(str)

    existing_files = {p.name for p in CYCLE_DATA_DIR.glob("*.csv")}

    discharge_meta = metadata.loc[metadata["type"] == "discharge"].copy()
    discharge_meta = discharge_meta.dropna(subset=["battery_id", "start_time", "Capacity", "filename"])
    discharge_meta = discharge_meta[(discharge_meta["Capacity"] > 0) & discharge_meta["filename"].isin(existing_files)].copy()
    discharge_meta = discharge_meta.sort_values(["battery_id", "start_time", "uid"], kind="mergesort")

    charge_meta = metadata.loc[metadata["type"] == "charge"].copy()
    charge_meta = charge_meta.dropna(subset=["battery_id", "start_time", "filename"])
    charge_meta = charge_meta[charge_meta["filename"].isin(existing_files)].copy()
    charge_meta = charge_meta.sort_values(["battery_id", "start_time", "uid"], kind="mergesort")

    print("Extracting discharge features:", len(discharge_meta))
    discharge_features = pd.DataFrame([extract_cycle_features(CYCLE_DATA_DIR / filename, prefix="d_") for filename in discharge_meta["filename"].tolist()])
    print("Extracting charge features:", len(charge_meta))
    charge_features = pd.DataFrame([extract_cycle_features(CYCLE_DATA_DIR / filename, prefix="c_") for filename in charge_meta["filename"].tolist()])

    discharge_df = discharge_meta.merge(discharge_features, on="filename", how="left")
    charge_df = charge_meta.merge(charge_features, on="filename", how="left")

    charge_cols = [c for c in charge_df.columns if c.startswith("c_")]
    aligned_parts = []
    for battery_id, left in discharge_df.groupby("battery_id"):
        right = charge_df.loc[charge_df["battery_id"] == battery_id, ["start_time"] + charge_cols].sort_values("start_time")
        left_sorted = left.sort_values("start_time")
        if right.empty:
            aligned_parts.append(left_sorted)
        else:
            aligned_parts.append(pd.merge_asof(left_sorted, right, on="start_time", direction="backward"))
    model_df = pd.concat(aligned_parts, ignore_index=True)

    impedance = metadata.loc[metadata["type"] == "impedance"].copy()
    impedance = impedance[(impedance["Re"] > 0) & (impedance["Re"] <= 10) & (impedance["Rct"] > 0) & (impedance["Rct"] <= 10)].copy()
    impedance = impedance.rename(columns={"Re": "imp_Re", "Rct": "imp_Rct", "start_time": "imp_start_time"})
    imp_parts = []
    for battery_id, left in model_df.groupby("battery_id"):
        right = impedance.loc[impedance["battery_id"] == battery_id, ["imp_start_time", "imp_Re", "imp_Rct"]].sort_values("imp_start_time")
        left_sorted = left.sort_values("start_time")
        if right.empty:
            left_sorted["imp_start_time"] = pd.NaT
            left_sorted["imp_Re"] = np.nan
            left_sorted["imp_Rct"] = np.nan
            imp_parts.append(left_sorted)
        else:
            imp_parts.append(pd.merge_asof(left_sorted, right, left_on="start_time", right_on="imp_start_time", direction="backward"))
    model_df = pd.concat(imp_parts, ignore_index=True)
    model_df["imp_age_hours"] = (model_df["start_time"] - model_df["imp_start_time"]).dt.total_seconds() / 3600.0

    def add_targets(group: pd.DataFrame) -> pd.DataFrame:
        g = group.sort_values("start_time").copy()
        g["battery_id"] = group.name
        g["cycle_index"] = np.arange(1, len(g) + 1)
        capacity = g["Capacity"].astype(float)
        positive = capacity[capacity > 0]
        median = float(positive.median()) if len(positive) else np.nan
        lower = 0.5 * median if np.isfinite(median) else 0.0
        upper = 1.4 * median if np.isfinite(median) else np.inf
        cap_clean = capacity.where((capacity >= lower) & (capacity <= upper), np.nan)

        # Several NASA groups contain a startup discharge with a capacity far from
        # the early stable plateau. Keeping that point makes SOH/EOL appear to be
        # crossed at cycle 1, so replace only that startup anomaly before interpolation.
        if len(cap_clean) >= 5:
            early_ref = float(cap_clean.iloc[1 : min(8, len(cap_clean))].dropna().median())
            if np.isfinite(early_ref) and early_ref > 0 and pd.notna(cap_clean.iloc[0]):
                first_ratio = float(cap_clean.iloc[0] / early_ref)
                if first_ratio < 0.85 or first_ratio > 1.15:
                    cap_clean.iloc[0] = np.nan

        cap_clean = cap_clean.interpolate(limit_direction="both").ffill().bfill()
        initial_capacity = float(cap_clean.head(min(5, len(cap_clean))).median())
        g["capacity_ah_clean"] = cap_clean
        g["initial_capacity"] = initial_capacity
        g["capacity_ratio"] = cap_clean / initial_capacity if initial_capacity > 0 else np.nan
        g["soh"] = g["capacity_ratio"]
        g["rul_cycles"] = (len(g) - g["cycle_index"]).astype(float)
        g["cap_ratio_delta1"] = g["capacity_ratio"].diff().fillna(0.0)
        g["cap_ratio_rollmean5"] = g["capacity_ratio"].rolling(5, min_periods=1).mean()
        g["cap_ratio_rollstd5"] = g["capacity_ratio"].rolling(5, min_periods=2).std().fillna(0.0)
        g["cap_ratio_slope5"] = rolling_slope(g["capacity_ratio"], window=5).fillna(0.0)
        return g

    model_df = model_df.groupby("battery_id", group_keys=False).apply(add_targets)
    model_df = model_df.dropna(subset=["battery_id", "cycle_index", "rul_cycles", "capacity_ratio"])
    counts = model_df.groupby("battery_id").size()
    eligible_batteries = counts[counts >= 20].index.tolist()
    model_df = model_df[model_df["battery_id"].isin(eligible_batteries)].copy()
    model_df = model_df.sort_values(["battery_id", "cycle_index"], kind="mergesort")
    return model_df

if FEATURE_TABLE_PATH.exists():
    model_df = pd.read_csv(FEATURE_TABLE_PATH, parse_dates=["start_time", "imp_start_time"])
    print("Loaded cached feature table:", FEATURE_TABLE_PATH)
else:
    model_df = build_feature_table()
    model_df.to_csv(FEATURE_TABLE_PATH, index=False)
    print("Saved feature table:", FEATURE_TABLE_PATH)

print("Rows:", len(model_df))
print("Batteries:", model_df["battery_id"].nunique())
display(model_df.groupby("battery_id").size().rename("rows").reset_index().sort_values("rows", ascending=False))

## 4. Feature set

In [ ]:
TARGET_COL = "rul_cycles"
GROUP_COL = "battery_id"

candidate_features = [
    "ambient_temperature", "cycle_index", "capacity_ah_clean", "capacity_ratio",
    "cap_ratio_delta1", "cap_ratio_rollmean5", "cap_ratio_rollstd5", "cap_ratio_slope5",
    "d_time_duration", "d_voltage_mean", "d_voltage_std", "d_voltage_min", "d_voltage_max", "d_voltage_slope",
    "d_current_mean", "d_current_std", "d_current_min", "d_current_max",
    "d_temperature_mean", "d_temperature_std", "d_temperature_max", "d_temperature_slope",
    "d_energy_wh", "d_charge_ah",
    "c_time_duration", "c_voltage_mean", "c_current_mean", "c_temperature_mean",
    "imp_Re", "imp_Rct", "imp_age_hours",
]
feature_cols = [c for c in candidate_features if c in model_df.columns]
FEATURE_COLUMNS_PATH.write_text(json.dumps({"target_col": TARGET_COL, "group_col": GROUP_COL, "feature_cols": feature_cols}, indent=2))
print("Feature count:", len(feature_cols))
print(feature_cols)

## 5. Battery quality table si scenarii de modelare

In [ ]:
def build_battery_quality_table(df: pd.DataFrame) -> pd.DataFrame:
    q = (
        df.groupby("battery_id")
        .agg(
            rows=("cycle_index", "max"),
            initial_capacity=("initial_capacity", "first"),
            final_capacity=("capacity_ah_clean", "last"),
            min_soh=("soh", "min"),
            max_soh=("soh", "max"),
            final_soh=("soh", "last"),
        )
        .reset_index()
    )
    q["soh_span"] = q["max_soh"] - q["min_soh"]
    q["clean_benchmark"] = (
        (q["rows"] >= 60)
        & (q["initial_capacity"] >= 0.5)
        & (q["max_soh"] <= 1.20)
        & (q["final_soh"] <= 1.05)
        & (q["min_soh"] <= 0.98)
    )
    q["exclusion_reason"] = ""
    q.loc[q["rows"] < 60, "exclusion_reason"] += "short_history;"
    q.loc[q["initial_capacity"] < 0.5, "exclusion_reason"] += "invalid_initial_capacity;"
    q.loc[q["max_soh"] > 1.20, "exclusion_reason"] += "soh_spike;"
    q.loc[q["final_soh"] > 1.05, "exclusion_reason"] += "capacity_increases_over_life;"
    q.loc[q["min_soh"] > 0.98, "exclusion_reason"] += "weak_observed_degradation;"
    q.loc[q["clean_benchmark"], "exclusion_reason"] = "included"
    return q.sort_values(["clean_benchmark", "rows", "battery_id"], ascending=[False, False, True])

quality = build_battery_quality_table(model_df)
quality.to_csv(QUALITY_TABLE_PATH, index=False)
print("Saved quality table:", QUALITY_TABLE_PATH)
display(quality)

clean_batteries = sorted(quality.loc[quality["clean_benchmark"], "battery_id"].tolist())
all_batteries = sorted(model_df["battery_id"].unique().tolist())
print("All eligible batteries:", len(all_batteries), all_batteries)
print("Clean benchmark batteries:", len(clean_batteries), clean_batteries)

## 6. Splituri pe baterii

In [ ]:
def make_group_split(df: pd.DataFrame, split_path: Path, seed: int, test_size: float = 0.20, val_size_of_trainval: float = 0.25, fixed_split: dict | None = None) -> dict:
    if fixed_split is not None:
        split = {
            "seed": seed,
            "strategy": "Fixed battery split",
            "train_batteries": fixed_split["train_batteries"],
            "validation_batteries": fixed_split["validation_batteries"],
            "test_batteries": fixed_split["test_batteries"],
        }
        split_path.write_text(json.dumps(split, indent=2))
        return split

    if split_path.exists():
        split = json.loads(split_path.read_text())
        # Keep reproducibility: do not silently overwrite existing splits.
        return split

    groups = df[GROUP_COL]
    gss_test = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    trainval_idx, test_idx = next(gss_test.split(df, df[TARGET_COL], groups=groups))
    trainval_df = df.iloc[trainval_idx]

    gss_val = GroupShuffleSplit(n_splits=1, test_size=val_size_of_trainval, random_state=seed + 1)
    train_idx_rel, val_idx_rel = next(gss_val.split(trainval_df, trainval_df[TARGET_COL], groups=trainval_df[GROUP_COL]))

    split = {
        "seed": seed,
        "strategy": "GroupShuffleSplit by battery_id",
        "test_size": test_size,
        "val_size_of_trainval": val_size_of_trainval,
        "train_batteries": sorted(trainval_df.iloc[train_idx_rel][GROUP_COL].unique().tolist()),
        "validation_batteries": sorted(trainval_df.iloc[val_idx_rel][GROUP_COL].unique().tolist()),
        "test_batteries": sorted(df.iloc[test_idx][GROUP_COL].unique().tolist()),
    }
    split_path.write_text(json.dumps(split, indent=2))
    return split

scenario_inputs = {
    "all_eligible": {
        "description": "Strict benchmark: all batteries with at least 20 usable discharge cycles.",
        "battery_ids": all_batteries,
        "seed": SEED,
        "split_path": SPLIT_DIR / "battery_split_all_eligible_v1.json",
    },
    "clean_benchmark": {
        "description": "Main thesis benchmark: batteries with enough cycles and coherent capacity/SOH behavior.",
        "battery_ids": clean_batteries,
        "seed": CLEAN_SPLIT_SEED,
        "split_path": SPLIT_DIR / "battery_split_clean_benchmark_v1.json",
    },
    "nasa_classic_4": {
        "description": "Classic NASA benchmark with B0005/B0006/B0007/B0018, useful for comparison with many public repositories.",
        "battery_ids": ["B0005", "B0006", "B0007", "B0018"],
        "seed": SEED,
        "split_path": SPLIT_DIR / "battery_split_nasa_classic_4_v1.json",
        "fixed_split": {
            "train_batteries": ["B0005", "B0006"],
            "validation_batteries": ["B0018"],
            "test_batteries": ["B0007"],
        },
    },
}

scenarios = {}
for scenario_name, cfg in scenario_inputs.items():
    scenario_df = model_df[model_df[GROUP_COL].isin(cfg["battery_ids"])].copy()
    split = make_group_split(scenario_df, cfg["split_path"], seed=cfg["seed"], fixed_split=cfg.get("fixed_split"))
    # Backward compatibility with the first notebook version.
    if scenario_name == "all_eligible":
        legacy_path = SPLIT_DIR / "battery_split_v1.json"
        if not legacy_path.exists():
            legacy_path.write_text(json.dumps(split, indent=2))
    scenarios[scenario_name] = {
        "description": cfg["description"],
        "battery_ids": cfg["battery_ids"],
        "split_path": str(cfg["split_path"].relative_to(REPO_ROOT)),
        "split": split,
    }

SCENARIO_CONFIG_PATH.write_text(json.dumps(scenarios, indent=2))
print("Saved scenario config:", SCENARIO_CONFIG_PATH)

rows = []
for name, cfg in scenarios.items():
    split = cfg["split"]
    for split_name, ids_key in [("train", "train_batteries"), ("validation", "validation_batteries"), ("test", "test_batteries")]:
        ids = split[ids_key]
        sub = model_df[model_df[GROUP_COL].isin(ids)]
        rows.append({"scenario": name, "split": split_name, "batteries": len(ids), "rows": len(sub), "battery_ids": ", ".join(ids)})
split_summary = pd.DataFrame(rows)
display(split_summary)

## 7. Modele baseline si evaluare pe scenarii

In [ ]:
def make_scaled_preprocessor() -> Pipeline:
    return Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])


def make_tree_preprocessor() -> Pipeline:
    return Pipeline([("imputer", SimpleImputer(strategy="median"))])

models = {
    "SVR_RBF": Pipeline([("prep", make_scaled_preprocessor()), ("model", SVR(kernel="rbf", C=30.0, epsilon=3.0, gamma="scale"))]),
    "RandomForest": Pipeline([("prep", make_tree_preprocessor()), ("model", RandomForestRegressor(n_estimators=500, min_samples_leaf=3, random_state=SEED, n_jobs=1))]),
    "ExtraTrees": Pipeline([("prep", make_tree_preprocessor()), ("model", ExtraTreesRegressor(n_estimators=500, min_samples_leaf=2, random_state=SEED, n_jobs=1))]),
    "HistGradientBoosting": Pipeline([("prep", make_tree_preprocessor()), ("model", HistGradientBoostingRegressor(max_iter=400, learning_rate=0.05, l2_regularization=0.1, random_state=SEED))]),
}

validation_frames = []
test_frames = []
prediction_frames = []
experiment_summaries = {}

for scenario_name, cfg in scenarios.items():
    split = cfg["split"]
    scenario_df = model_df[model_df[GROUP_COL].isin(cfg["battery_ids"])].copy()
    train_df = scenario_df[scenario_df[GROUP_COL].isin(split["train_batteries"])].copy()
    val_df = scenario_df[scenario_df[GROUP_COL].isin(split["validation_batteries"])].copy()
    test_df = scenario_df[scenario_df[GROUP_COL].isin(split["test_batteries"])].copy()
    trainval_df = scenario_df[scenario_df[GROUP_COL].isin(split["train_batteries"] + split["validation_batteries"])].copy()

    X_train, y_train = train_df[feature_cols], train_df[TARGET_COL]
    X_val, y_val = val_df[feature_cols], val_df[TARGET_COL]
    X_trainval, y_trainval = trainval_df[feature_cols], trainval_df[TARGET_COL]
    X_test, y_test = test_df[feature_cols], test_df[TARGET_COL]

    val_rows = []
    for model_name, estimator in models.items():
        model = clone(estimator)
        model.fit(X_train, y_train)
        pred_val = model.predict(X_val)
        val_rows.append({"scenario": scenario_name, "model": model_name, "split": "validation", **regression_metrics(y_val, pred_val)})

    naive_val = np.full_like(y_val.to_numpy(dtype=float), fill_value=float(y_train.mean()), dtype=float)
    val_rows.append({"scenario": scenario_name, "model": "NaiveTrainMean", "split": "validation", **regression_metrics(y_val, naive_val)})
    val_metrics = pd.DataFrame(val_rows).sort_values("RMSE")
    validation_frames.append(val_metrics)

    best_by_validation = val_metrics.loc[val_metrics["model"] != "NaiveTrainMean", "model"].iloc[0]

    test_rows = []
    scenario_predictions = []
    for model_name, estimator in models.items():
        model = clone(estimator)
        model.fit(X_trainval, y_trainval)
        pred_test = model.predict(X_test)
        test_rows.append({"scenario": scenario_name, "model": model_name, "split": "test", **regression_metrics(y_test, pred_test)})

        pred_frame = test_df[["battery_id", "cycle_index", "start_time", TARGET_COL, "capacity_ah_clean", "capacity_ratio"]].copy()
        pred_frame["scenario"] = scenario_name
        pred_frame["model"] = model_name
        pred_frame["prediction"] = pred_test
        pred_frame["error"] = pred_frame["prediction"] - pred_frame[TARGET_COL]
        pred_frame["abs_error"] = pred_frame["error"].abs()
        scenario_predictions.append(pred_frame)

        model_path = MODEL_DIR / f"baseline_{scenario_name}_{model_name.lower()}_trainval.joblib"
        joblib.dump(model, model_path)

    naive_test = np.full_like(y_test.to_numpy(dtype=float), fill_value=float(y_trainval.mean()), dtype=float)
    test_rows.append({"scenario": scenario_name, "model": "NaiveTrainValMean", "split": "test", **regression_metrics(y_test, naive_test)})
    test_metrics = pd.DataFrame(test_rows).sort_values("RMSE")
    test_frames.append(test_metrics)
    prediction_frames.append(pd.concat(scenario_predictions, ignore_index=True))

    best_by_test = test_metrics.loc[test_metrics["model"] != "NaiveTrainValMean", "model"].iloc[0]
    deployment_model = clone(models[best_by_validation])
    deployment_model.fit(scenario_df[feature_cols], scenario_df[TARGET_COL])
    deployment_path = MODEL_DIR / f"baseline_{scenario_name}_{best_by_validation.lower()}_all_data_deployment.joblib"
    joblib.dump(deployment_model, deployment_path)

    experiment_summaries[scenario_name] = {
        "description": cfg["description"],
        "best_validation_model": best_by_validation,
        "best_test_model_diagnostic": best_by_test,
        "deployment_model": str(deployment_path.relative_to(REPO_ROOT)),
        "split": split,
    }

validation_metrics = pd.concat(validation_frames, ignore_index=True).sort_values(["scenario", "RMSE"])
test_metrics = pd.concat(test_frames, ignore_index=True).sort_values(["scenario", "RMSE"])
predictions = pd.concat(prediction_frames, ignore_index=True)

validation_metrics.to_csv(METRICS_DIR / "baseline_validation_metrics.csv", index=False)
test_metrics.to_csv(METRICS_DIR / "baseline_test_metrics.csv", index=False)
predictions.to_csv(PRED_DIR / "baseline_test_predictions.csv", index=False)
(METRICS_DIR / "baseline_experiment_summary.json").write_text(json.dumps(experiment_summaries, indent=2))

print("Validation metrics:")
display(validation_metrics)
print("Test metrics:")
display(test_metrics)

 ## 8. Grafice baseline pe scenarii

In [ ]:
plot_df = test_metrics[~test_metrics["model"].str.contains("Naive", case=False, na=False)].copy()
fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
for ax, scenario_name in zip(axes, ["all_eligible", "clean_benchmark"]):
    sub = plot_df[plot_df["scenario"] == scenario_name].sort_values("RMSE")
    ax.bar(sub["model"], sub["RMSE"], color="#3a6ea5")
    ax.set_title(f"{scenario_name}: test RMSE")
    ax.set_xlabel("Model")
    ax.set_ylabel("RMSE (cycles)")
    ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
savefig("01_baseline_rmse_by_scenario.png")
plt.show()

for scenario_name, summary in experiment_summaries.items():
    selected_model = summary["best_validation_model"]
    pred_df = predictions[(predictions["scenario"] == scenario_name) & (predictions["model"] == selected_model)].copy()
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].scatter(pred_df[TARGET_COL], pred_df["prediction"], s=18, alpha=0.55)
    line_min = float(min(pred_df[TARGET_COL].min(), pred_df["prediction"].min()))
    line_max = float(max(pred_df[TARGET_COL].max(), pred_df["prediction"].max()))
    axes[0].plot([line_min, line_max], [line_min, line_max], color="red", linestyle="--", linewidth=1)
    axes[0].set_title(f"{scenario_name}: {selected_model} predicted vs true")
    axes[0].set_xlabel("True RUL (cycles)")
    axes[0].set_ylabel("Predicted RUL (cycles)")
    axes[1].hist(pred_df["error"], bins=35, color="#3a6ea5")
    axes[1].axvline(0, color="red", linestyle="--", linewidth=1)
    axes[1].set_title("Prediction error distribution")
    axes[1].set_xlabel("Prediction error (cycles)")
    plt.tight_layout()
    savefig(f"02_{scenario_name}_selected_baseline_pred_vs_true.png")
    plt.show()

## 9. Eroare pe baterii si curbe RUL

In [ ]:
def per_battery_metrics(df: pd.DataFrame, true_col: str, pred_col: str) -> pd.DataFrame:
    rows = []
    for (scenario_name, model_name, battery_id), g in df.groupby(["scenario", "model", "battery_id"]):
        rows.append({"scenario": scenario_name, "model": model_name, "battery_id": battery_id, "rows": len(g), **regression_metrics(g[true_col], g[pred_col])})
    return pd.DataFrame(rows).sort_values(["scenario", "model", "MAE"])

battery_metrics = per_battery_metrics(predictions, TARGET_COL, "prediction")
battery_metrics.to_csv(METRICS_DIR / "baseline_per_battery_test_metrics.csv", index=False)

for scenario_name, summary in experiment_summaries.items():
    selected_model = summary["best_validation_model"]
    selected = predictions[(predictions["scenario"] == scenario_name) & (predictions["model"] == selected_model)].copy()
    bm = battery_metrics[(battery_metrics["scenario"] == scenario_name) & (battery_metrics["model"] == selected_model)].sort_values("MAE")
    print("\n", scenario_name, selected_model)
    display(bm)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(bm["battery_id"], bm["MAE"], color="#3a6ea5")
    ax.set_title(f"{scenario_name}: {selected_model} test MAE by battery")
    ax.set_xlabel("Battery ID")
    ax.set_ylabel("MAE (cycles)")
    ax.tick_params(axis="x", rotation=30)
    plt.tight_layout()
    savefig(f"03_{scenario_name}_selected_baseline_mae_by_battery.png")
    plt.show()

    test_battery_ids = sorted(selected["battery_id"].unique())
    cols = 2
    rows_n = int(np.ceil(len(test_battery_ids) / cols))
    fig, axes = plt.subplots(rows_n, cols, figsize=(14, 4 * rows_n), squeeze=False)
    for ax, battery_id in zip(axes.ravel(), test_battery_ids):
        g = selected[selected["battery_id"] == battery_id].sort_values("cycle_index")
        ax.plot(g["cycle_index"], g[TARGET_COL], label="true RUL", linewidth=2)
        ax.plot(g["cycle_index"], g["prediction"], label="predicted RUL", linewidth=1.6)
        ax.set_title(battery_id)
        ax.set_xlabel("Cycle index")
        ax.set_ylabel("RUL (cycles)")
        ax.legend(fontsize=8)
    for ax in axes.ravel()[len(test_battery_ids):]:
        ax.axis("off")
    plt.tight_layout()
    savefig(f"04_{scenario_name}_selected_baseline_rul_curves_by_test_battery.png")
    plt.show()

## 10. Concluzie pentru notebookul 03

Notebookul 03 foloseste `modeling_scenarios_v1.json`, aceeasi tabela de feature-uri si aceleasi splituri pentru a antrena LSTM/CNN-LSTM in ambele scenarii.